In [0]:
%sql
WITH m AS (SELECT * FROM laddstolpar_df.gold.mart_kommun_metric),
w AS (
  SELECT a.kommun_kod,
         a.pct_rank    AS g_shrunk,
         r.pct_rank    AS g_raw,
         d.pct_rank    AS demand,
         a.denominator AS n_new_cars,
         a.value       AS d_shrunk,
         r.value       AS d_raw
  FROM m a
  JOIN m r USING (kommun_kod)
  JOIN m d USING (kommun_kod)
  WHERE a.metric = 'growth' AND r.metric = 'growth_raw' AND d.metric = 'ev_demand'
)
SELECT
  ROUND(corr(g_raw, demand), 2)                         AS rho_raw_demand,
  ROUND(corr(g_shrunk, demand), 2)                      AS rho_shrunk_demand,
  ROUND(corr(g_shrunk, g_raw), 2)                       AS rho_shrunk_raw,
  CAST(percentile(n_new_cars, 0.5) AS BIGINT)           AS median_new_cars,
  COUNT_IF(n_new_cars < 46)                             AS kommuner_below_k,
  ROUND(100 * AVG(ABS(d_shrunk - d_raw)), 2)            AS mean_shift_pp,
  ROUND(100 * MAX(ABS(d_shrunk - d_raw)), 2)            AS max_shift_pp
FROM w;

In [0]:
%sql
WITH a AS (
  SELECT a.*, k.kommun_namn
  FROM laddstolpar_df.gold.mart_allocation a
  JOIN laddstolpar_df.gold.dim_kommun k USING (kommun_kod)
),
m AS (SELECT * FROM laddstolpar_df.gold.mart_kommun_metric)
SELECT
  (SELECT COUNT(*) FROM laddstolpar_df.gold.mart_kommun_score_component)                   AS component_rows,
  (SELECT COUNT(*) FROM laddstolpar_df.gold.mart_kommun_score)                             AS score_rows,
  (SELECT COUNT(DISTINCT scenario, variant) FROM a)                                        AS runs,
  (SELECT CAST(MIN(t) AS STRING) || '-' || CAST(MAX(t) AS STRING)
     FROM (SELECT SUM(n_stations) AS t FROM a GROUP BY scenario, variant))                 AS stations_per_run,
  (SELECT MAX(n_stations) FROM a)                                                          AS max_in_one_kommun,
  (SELECT concat_ws(', ', transform(array_sort(collect_list(struct(-n_stations AS s, score_rank AS r,
            kommun_namn || ' ' || CAST(n_stations AS STRING) AS t))), x -> x.t))
     FROM a WHERE scenario = 'S1' AND variant = 'base' AND n_stations > 0)                 AS s1_base,
  (SELECT concat_ws(', ', transform(array_sort(collect_list(struct(-n_stations AS s, score_rank AS r,
            kommun_namn || ' ' || CAST(n_stations AS STRING) AS t))), x -> x.t))
     FROM a WHERE scenario = 'S2' AND variant = 'base' AND n_stations > 0)                 AS s2_base,
  (SELECT concat_ws(', ', transform(array_sort(collect_list(struct(-n_stations AS s, score_rank AS r,
            kommun_namn || ' ' || CAST(n_stations AS STRING) AS t))), x -> x.t))
     FROM a WHERE scenario = 'S3' AND variant = 'base' AND n_stations > 0)                 AS s3_base,
  (SELECT COUNT(*) FROM (SELECT kommun_kod FROM a WHERE variant = 'base' AND n_stations > 0
     GROUP BY kommun_kod HAVING COUNT(*) = 3))                                             AS robust_in_all_3,
  (SELECT ROUND(corr(x.pct_rank, y.pct_rank), 2) FROM m x JOIN m y USING (kommun_kod)
     WHERE x.metric = 'growth_stock' AND y.metric = 'ev_demand')                           AS rho_c_demand,
  (SELECT ROUND(corr(x.pct_rank, y.pct_rank), 2) FROM m x JOIN m y USING (kommun_kod)
     WHERE x.metric = 'growth' AND y.metric = 'growth_stock')                              AS rho_d_c;

In [0]:
%sql
WITH a AS (
  SELECT * FROM laddstolpar_df.gold.mart_allocation WHERE variant IN ('base', 'no_cost')
),
thr AS (
  SELECT scenario, variant, MIN(quotient) AS q50
  FROM laddstolpar_df.gold.mart_allocation_station
  GROUP BY scenario, variant
)
SELECT a.scenario, a.variant,
       ROUND(MAX(t.q50), 3)                                              AS quotient_50th,
       ROUND(MAX(a.score) / 3, 3)                                        AS best_possible_2nd_station,
       ROUND(MAX(CASE WHEN a.kommun_kod = '0180' THEN a.score END), 3)   AS sthlm_score,
       MAX(CASE WHEN a.kommun_kod = '0180' THEN a.score_rank END)        AS sthlm_rank,
       MAX(CASE WHEN a.kommun_kod = '0180' THEN a.n_stations END)        AS sthlm_n,
       MAX(CASE WHEN a.kommun_kod = '1480' THEN a.score_rank END)        AS gbg_rank,
       MAX(CASE WHEN a.kommun_kod = '1480' THEN a.n_stations END)        AS gbg_n,
       MAX(CASE WHEN a.kommun_kod = '1280' THEN a.score_rank END)        AS malmo_rank,
       MAX(CASE WHEN a.kommun_kod = '1280' THEN a.n_stations END)        AS malmo_n
FROM a
JOIN thr t USING (scenario, variant)
GROUP BY a.scenario, a.variant
ORDER BY a.scenario, a.variant;

In [0]:
G = "laddstolpar_df.gold"

# 1. Structure – expect 3480, 12, 14, 700, and a low or negative rho
display(spark.sql(f"""
SELECT 'metric rows' AS what, CAST(COUNT(*) AS DOUBLE) AS n FROM {G}.mart_kommun_metric
UNION ALL SELECT 'metrics', CAST(COUNT(DISTINCT metric) AS DOUBLE) FROM {G}.mart_kommun_metric
UNION ALL SELECT 'runs', CAST(COUNT(DISTINCT scenario, variant) AS DOUBLE) FROM {G}.mart_kommun_score
UNION ALL SELECT 'stations allocated', CAST(COUNT(*) AS DOUBLE) FROM {G}.mart_allocation_station
UNION ALL SELECT 'rho through_traffic vs ev_demand', ROUND(corr(t.pct_rank, e.pct_rank), 3)
  FROM {G}.mart_kommun_metric t JOIN {G}.mart_kommun_metric e USING (kommun_kod)
  WHERE t.metric = 'through_traffic' AND e.metric = 'ev_demand'
"""))

# 2. Per run: spread and the big three (H1)
display(spark.sql(f"""
SELECT scenario, variant,
       COUNT_IF(n_stations > 0)                          AS n_kommuner,
       MAX(n_stations)                                   AS max_in_one,
       SUM(CASE WHEN kommun_kod = '0180' THEN n_stations END) AS stockholm,
       SUM(CASE WHEN kommun_kod = '1480' THEN n_stations END) AS goteborg,
       SUM(CASE WHEN kommun_kod = '1280' THEN n_stations END) AS malmo
FROM {G}.mart_allocation GROUP BY scenario, variant ORDER BY scenario, variant
"""))

# 3. Base-run picks for H1-H3
display(spark.sql(f"""
SELECT a.scenario, k.kommun_namn, a.n_stations, a.first_seat,
       ROUND(a.attractiveness, 3) AS attr, a.quantity, ROUND(a.quantity_value) AS q, ROUND(a.votes) AS votes,
       k.kommungrupp, k.is_besoksnaring AS tourist, k.elomrade
FROM {G}.mart_allocation a JOIN {G}.dim_kommun k USING (kommun_kod)
WHERE a.variant = 'base' AND a.n_stations > 0
ORDER BY a.scenario, a.n_stations DESC, a.first_seat
"""))

# 4. Top 15 through-traffic (finding 72 follow-up)
display(spark.sql(f"""
SELECT k.kommun_namn, ROUND(m.value, 1) AS vkm_per_inhabitant, ROUND(m.pct_rank, 3) AS pct_rank, k.kommungrupp
FROM {G}.mart_kommun_metric m JOIN {G}.dim_kommun k USING (kommun_kod)
WHERE m.metric = 'through_traffic' ORDER BY m.value DESC LIMIT 15
"""))

# 5. score H2/H3 directly
display(spark.sql(f"""
SELECT a.scenario, a.variant,
       COUNT_IF(a.n_stations > 0 AND k.is_besoksnaring)                              AS h2_tourist_kommuner,
       COUNT_IF(a.n_stations > 0 AND m.pct_rank >= 0.75)                             AS h3_top_quartile_through,
       COUNT_IF(a.n_stations > 0 AND k.kommungrupp LIKE 'Pendlingskommun nära storstad') AS metro_suburbs
FROM {G}.mart_allocation a
JOIN {G}.dim_kommun k USING (kommun_kod)
JOIN {G}.mart_kommun_metric m ON m.kommun_kod = a.kommun_kod AND m.metric = 'through_traffic'
WHERE a.variant IN ('base', 'quantity_ev', 'alpha_025', 'alpha_075')
GROUP BY a.scenario, a.variant ORDER BY a.scenario, a.variant
"""))